# Step 4: Create Dashboard

Runs after `create_space`. Calls the LLM to design dashboard panels,
executes each panel's SQL, and stores the results in the sessions table.

In [ ]:
%pip install openai
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "yd_launchpad_final_classic_catalog")
dbutils.widgets.text("schema", "genie_app")
dbutils.widgets.text("company_name", "NovaTech Logistics")

import re
def slugify(name: str) -> str:
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', name.lower()).strip('_')
    if slug and slug[0].isdigit():
        slug = f'co_{slug}'
    return slug[:50]

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
COMPANY_NAME = dbutils.widgets.get("company_name")
COMPANY_SLUG = slugify(COMPANY_NAME)
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/{COMPANY_SLUG}"

print(f"Company: {COMPANY_NAME}")
print(f"Volume path: {VOLUME_PATH}")

In [ ]:
import json
import uuid
from datetime import datetime, timezone

# Load pipeline state from previous steps
state_raw = dbutils.fs.head(f"{VOLUME_PATH}/pipeline_state.json", 100000)
state = json.loads(state_raw)

COMPANY_DESC = state["company_description"]
tables_info = state["tables_info"]
space_id = state.get("space_id", "")

# Get workspace host and token for LLM call
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()
DATABRICKS_HOST_ID = "7474655921234161"
LLM_MODEL = "opendoor-claude-opus-46"

print(f"Space ID: {space_id}")
print(f"Tables: {len(tables_info)}")

In [ ]:
# Design dashboard panels via LLM (inlined to avoid package import path issues)
import re
from openai import OpenAI

DASHBOARD_SYSTEM_PROMPT = """You are a data dashboard designer. Given table schemas for a company's analytics database, generate 4-6 dashboard panel definitions.

RULES:
1. Start with 1-2 KPI panels (single aggregate number), then categorical charts, then time series.
2. Each panel's SQL must use fully-qualified 3-part table names (catalog.schema.table).
3. Keep SQL simple — single table queries, basic aggregates, GROUP BY. MAX 20 rows per query.
4. chart_type must be one of: kpi, bar, line, pie, area
5. For KPI: SQL should return exactly 1 row with 1 numeric column.
6. For bar/pie: SQL should return a categorical column + a numeric column.
7. For line/area: SQL should return a date/time column + a numeric column, ORDER BY the date column.
8. Position panels logically: KPIs first (position 0,1), then charts (position 2,3,4,5).

OUTPUT FORMAT (strict JSON array, no markdown fences):
[
  {
    "title": "Total Revenue",
    "sql": "SELECT SUM(revenue) as total_revenue FROM catalog.schema.table",
    "chart_type": "kpi",
    "position": 0
  },
  {
    "title": "Revenue by Category",
    "sql": "SELECT category, SUM(revenue) as revenue FROM catalog.schema.table GROUP BY category ORDER BY revenue DESC LIMIT 10",
    "chart_type": "bar",
    "position": 2
  }
]
"""

client = OpenAI(
    api_key=token,
    base_url=f"https://{DATABRICKS_HOST_ID}.ai-gateway.cloud.databricks.com/mlflow/v1",
)

# Build table schema description for the prompt
tables_desc = []
for t in tables_info:
    full_name = t.get("full_name", f"{CATALOG}.{SCHEMA}.{t.get('table_name', '')}")
    cols = t.get("columns", [])
    if cols:
        col_lines = [f"  - {c.get('name', '')}: {c.get('type', '')} ({c.get('comment', '')})" for c in cols]
        tables_desc.append(f"Table: {full_name}\nColumns:\n" + "\n".join(col_lines))
    else:
        tables_desc.append(f"Table: {full_name}")

prompt = (
    f"Company: {COMPANY_NAME}\n"
    f"Description: {COMPANY_DESC}\n"
    f"Catalog: {CATALOG}, Schema: {SCHEMA}\n\n"
    f"Tables:\n{'---'.join(tables_desc)}"
)

try:
    print("Calling LLM to design dashboard...")
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        max_tokens=4096,
        messages=[
            {"role": "system", "content": DASHBOARD_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    )

    raw = resp.choices[0].message.content.strip()
    # Strip markdown fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1]
        if raw.endswith("```"):
            raw = raw[:raw.rfind("```")]
    # Extract JSON array
    start = raw.find("[")
    end = raw.rfind("]") + 1
    if start >= 0 and end > start:
        raw = raw[start:end]
    # Fix trailing commas
    raw = re.sub(r",\s*([}\]])", r"\1", raw)

    panel_defs = json.loads(raw)
    if not isinstance(panel_defs, list):
        raise ValueError(f"Expected JSON array, got {type(panel_defs).__name__}")

    print(f"LLM generated {len(panel_defs)} panel definitions")
    for p in panel_defs:
        print(f"  - {p.get('title')} ({p.get('chart_type')})")
except Exception as e:
    print(f"ERROR: LLM dashboard design failed: {e}")
    panel_defs = []

In [ ]:
# Execute each panel's SQL and collect results
from datetime import date, datetime, timezone

panels = []
for i, pdef in enumerate(panel_defs):
    title = pdef.get("title", f"Panel {i}")
    sql = pdef.get("sql", "")
    chart_type = pdef.get("chart_type", "bar")
    position = pdef.get("position", i)

    if not sql:
        print(f"  SKIP '{title}': no SQL")
        continue

    try:
        df = spark.sql(sql)
        rows = df.limit(20).toPandas()
        columns = list(rows.columns)
        data = rows.to_dict(orient="records")
        # Convert numpy/pandas types to native Python
        for row in data:
            for k, v in row.items():
                if hasattr(v, "item"):
                    row[k] = v.item()
                elif v is None or (hasattr(v, "__class__") and "nat" in type(v).__name__.lower()):
                    row[k] = None
                elif isinstance(v, (date, datetime)):
                    row[k] = v.isoformat()

        panels.append({
            "id": uuid.uuid4().hex[:12],
            "title": title,
            "chart_type": chart_type,
            "sql": sql,
            "columns": columns,
            "data": data,
            "position": position,
        })
        print(f"  OK '{title}': {len(data)} rows, {len(columns)} cols")
    except Exception as e:
        print(f"  FAIL '{title}': {e}")

print(f"\n{len(panels)} of {len(panel_defs)} panels succeeded")

In [ ]:
# Store dashboard JSON in sessions table
sessions_table = f"`{CATALOG}`.`{SCHEMA}`.`sessions`"

# Add dashboard_json column if it doesn't exist
try:
    spark.sql(f"ALTER TABLE {sessions_table} ADD COLUMNS (dashboard_json STRING COMMENT 'Pre-computed dashboard JSON')")
    print("Added dashboard_json column")
except Exception:
    pass  # Column already exists

dashboard_payload = {
    "panels": panels,
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
dashboard_json = json.dumps(dashboard_payload).replace("'", "''")

# Find the session row for this space
safe_name = COMPANY_NAME.replace("'", "''")
spark.sql(f"""
    UPDATE {sessions_table}
    SET dashboard_json = '{dashboard_json}'
    WHERE company_name = '{safe_name}'
""")

print(f"Dashboard JSON stored for '{COMPANY_NAME}' ({len(panels)} panels)")

# Also update state.json
state["dashboard"] = dashboard_payload
state_json = json.dumps(state, indent=2)
dbutils.fs.put(f"{VOLUME_PATH}/pipeline_state.json", state_json, overwrite=True)

print(f"\nDashboard pipeline complete!")